# ViSceT5 — Pretrain **gen_all** (decoder read-scene-text, đòn bẩy #1)
Chạy tuần tự. `gen_all` = huấn luyện decoder **sinh scene-text** (khớp đúng đường finetune: encoder chỉ nhận câu hỏi + ảnh + OCR-feature) + MLM/ITM/TWC làm phụ trợ (×0.5) — phần pretrain trực tiếp có ích cho bộ sinh câu trả lời seq2seq.

Sau khi pretrain xong & upload lên HF, dùng `notebooks/finetune_colab.ipynb` để finetune từ nó.

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
# QUAN TRỌNG: các thay đổi pretrain (gen_all, vision unfreeze, whole-word mask) nằm ở
# NHÁNH exp/pretrain-gen-all — KHÔNG phải main. Không checkout đúng nhánh sẽ bị lỗi
# 'unrecognized arguments: --vision_unfreeze_last_n --mlm_mask_mode'.
!git fetch origin
!git checkout exp/pretrain-gen-all
!git pull origin exp/pretrain-gen-all
!git log --oneline -1

In [ ]:
%%capture
!bash setup.sh

In [ ]:
import os
HF_PRETRAIN_REPO = 'Kus669/ViSceT5-pretrain-genall'     # repo sẽ lưu MODEL PRETRAIN (gen_all)

In [ ]:
import argparse
from scripts import prepare_dataset
prepare_dataset.main(argparse.Namespace(config='configs/data/ViTextVQA.yaml', data_dir='./datasets'))

In [ ]:
from scripts import init_model
init_model.main()

### 1) SMOKE / MOCK — TỰ ĐỘNG in debug đầy đủ (không cần set env)
> ⚠️ Vision unfreeze TẮT, có guard NaN pretrain-only. ✅ MLM chỉ dùng câu hỏi. 📖 Gen = **read-scene-text** (denoise đã bỏ).

Mock **luôn** in debug. Trong log tìm:
1. `>>> [pretrain] ... gen = read-scene-text`
2. `🔬 [VERIFY]` toàn ✅, `✅ [GEN] gen_loss finite & > 0`, KHÔNG có `🚨 ... has NaN`
3. Per-step `[Pretrain] ... Loss(M) Loss(I) Loss(TWC) Loss(GEN)` **giảm dần**
4. `🔎 [MLM DEBUG]`: mỗi từ mask hiện **token thô gold/pred + gộp thành word**
5. `🔧 [GEN DEBUG]`: target (OCR reading) vs output

In [ ]:
import importlib
from training import pretrain
importlib.reload(pretrain)
# MOCK: tự động in debug đầy đủ (không cần set env). Full run mặc định KHÔNG debug.
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '0',   # TẮT tạm: vision unfreeze gây grad-nổ + NaN forward
    '--mlm_mask_mode', 'wholeword',    # mask trọn từ (bỏ 'nạng' copy subword)
    '--smoke_test', 'True',
])

### 2) FULL PRETRAIN — mặc định KHÔNG debug (chỉ progress bar + eval)
Full run **mặc định tắt debug** (chỉ thanh tiến trình train + kết quả eval trên val → tránh đầy output/lag). Muốn **bật debug** cho full: đặt `os.environ['TWC_TRAIN_LOG']='1'` trước khi gọi. Theo dõi `loss_mlm` & `loss_gen` **giảm dần** qua các lần eval.

In [ ]:
import os, importlib
os.environ.pop('TWC_TRAIN_LOG', None)   # TẮT debug per-step: full run chỉ hiện progress bar + kết quả eval
from training import pretrain
importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '0',   # TẮT tạm (vision instability)
    '--mlm_mask_mode', 'wholeword',
    '--num_train_epochs', '3',
])

### 2b) TRAIN THÊM (reproducible) — warm-start + CONSTANT LR
Sau khi đã xong N epoch (cosine LR về ~0), muốn train thêm mà **LR KHÔNG phụ thuộc số epoch**:
dùng **warm-start** (nạp weights, optimizer/scheduler MỚI) + **`lr_scheduler_type=constant`** →
LR cố định, không dính trạng thái cosine cũ, deterministic theo seed.
*(Muốn TRUE-resume bảo toàn momentum về sau: đặt `lr_scheduler_type: constant` NGAY TỪ ĐẦU ở configs/pretrain.yaml — khi đó resume_from_checkpoint với số epoch bất kỳ đều giữ LR cố định.)*

In [ ]:
# === TRAIN THÊM sau khi đã xong N epoch — warm-start + CONSTANT LR (reproducible) ===
import os, importlib
from huggingface_hub import snapshot_download

RESUME_REPO  = 'Kus669/ViSceT5-pretrain-genall'
CKPT         = 'checkpoint-6591'     # mốc CUỐI của run trước (đổi cho đúng)
OUT_DIR      = '/content/ViSceT5/output/pretrain'
EXTRA_EPOCHS = 2                     # số epoch train THÊM

snapshot_download(repo_id=RESUME_REPO, repo_type='model',
                  allow_patterns=[f'{CKPT}/*'], local_dir=OUT_DIR)
ckpt_path = os.path.join(OUT_DIR, CKPT)
print('Warm-start from:', ckpt_path)

# knob v2 (data-side) — GIỮ ĐÚNG như run gốc để độ khó data nhất quán
os.environ['TWC_ADV_PROB']='0.6'; os.environ['TWC_DUP_BOX']='0'
os.environ['MLM_RAND_PROB']='0.25'; os.environ['ITM_WEIGHT']='0'
os.environ.pop('TWC_TRAIN_LOG', None)

from training import pretrain; importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode','gen_all','--vision_unfreeze_last_n','0','--mlm_mask_mode','wholeword',
    '--model_name_or_path', ckpt_path,   # WARM-START: nạp weights; optimizer+scheduler MỚI
    '--lr_scheduler_type','constant',    # LR CỐ ĐỊNH -> KHÔNG phụ thuộc số epoch
    '--warmup_ratio','0.0',              # đã warm ở run trước
    '--learning_rate','3e-5',            # hạ nhẹ cho continuation (tránh sốc)
    '--num_train_epochs', str(EXTRA_EPOCHS),
    '--seed','42',                       # cố định seed -> reproducible
])

### 3) Upload model pretrain lên HF (để finetune_colab.ipynb dùng)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=HF_PRETRAIN_REPO, repo_type='model', exist_ok=True)
api.upload_folder(folder_path='/content/ViSceT5/output/pretrain', repo_id=HF_PRETRAIN_REPO,
                  repo_type='model', ignore_patterns=['optimizer.pt'])
print('Uploaded pretrain ->', HF_PRETRAIN_REPO)